# Mejoras del BSA y techo del traductor

Este notebook parte de la evaluación de `amt-bsa-goldberg` y busca subir la Exactitud de Símbolo Braille con ajustes ligeros sobre las notas que entrega el modelo acústico, sin entrenar nada nuevo. Para no ajustar a los datos de evaluación, las piezas de Goldberg se dividen una sola vez en una partición de desarrollo y otra de prueba. Todas las decisiones se toman con la de desarrollo, y la de prueba y un conjunto externo de tres obras se miden una única vez con la configuración ya elegida.

Al final se traduce la partitura digital de la misma edición directamente a Braille, sin audio. Ese número es lo más alto que puede dar el sistema con una transcripción perfecta, y separa la parte de la brecha que corresponde al traductor de la que corresponde a la transcripción.

## 1. Código y datos

In [1]:
import glob, io, json, sys, urllib.request, zipfile
from functools import partial
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from amt.postprocess import collapse_trills, drop_octave_ghosts, split_hands_by_continuity
from braille_translator.model import Score
from braille_translator.musicxml_parser import parse_musicxml
from braille_translator.translator import translate_score
from evaluation.bsa import compute_bsa
from evaluation.reference import parse_reference, reference_streams
from evaluation.reference_eval import MAX_CELLS, evaluate_piece, merge

NOTAS = ROOT / "results" / "oe5_notas"
NOTAS_EXT = ROOT / "results" / "oe5_externo_notas"
SALIDA = ROOT / "results"
DATOS = ROOT / "data"
DATOS.mkdir(exist_ok=True)

def descargar(url, destino):
    destino = DATOS / destino
    if not destino.exists():
        destino.write_bytes(urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "curl/8.0"})).read())
    return destino

goldberg = parse_reference(descargar("https://opengoldbergvariations.org/Bach-Goldberg-Variations-Braille-Edition.brf", "goldberg.brf").read_text("ascii"))
nombres = ["Aria"] + [f"Variación {i}" for i in range(1, 31)] + ["Aria da capo"]

## 2. Partición fija

Desarrollo son el Aria, las variaciones 1 a 15 y el Aria da capo. Prueba son las variaciones 16 a 30, sin la 25, que pasa del alcance de cinco minutos. La división sigue el orden de la obra y no se eligió mirando resultados.

In [2]:
DESARROLLO = list(range(16)) + [31]
PRUEBA = [i for i in range(16, 31) if i != 25]

def notas(modelo, i):
    return json.loads((NOTAS / f"{modelo}__{i:02d}.json").read_text())

def bsa_medio(piezas, modelo="ajustado", **kw):
    filas = []
    for i in piezas:
        d = notas(modelo, i)
        ev = evaluate_piece(goldberg[i], d["notas"], d["duracion_s"], **kw)
        filas.append((ev.both.bsa, ev.right.bsa, ev.left.bsa))
    return [sum(f[k] for f in filas) / len(filas) for k in range(3)]

## 3. Heurísticas

Cada una ataca un problema que ya se había visto al comparar compases generados con la referencia.

- H1 quita armónicos de octava. Si una nota suena una octava, una duodécima o dos octavas por encima de otra, empieza casi al mismo tiempo y no es más fuerte, se trata como un armónico que el modelo confundió con una nota.
- H2 reparte las manos por continuidad. En vez de cortar en el Do central, cada acorde se divide por el punto que deja a cada mano más cerca de su registro reciente.
- H3 colapsa trinos. Una alternancia rápida entre dos notas vecinas se escribe como la nota principal con la duración de todo el adorno.
- Voces separa la mano derecha en dos voces cuando una nota se sostiene mientras otras se mueven, y escribe la segunda con in-accord, como hace la edición.

El legato es la fracción del intervalo entre dos ataques que puede quedar en silencio y aun así escribirse como parte de la nota.

In [3]:
h1 = partial(drop_octave_ghosts, ratio=1.0, window_s=0.05)
h2 = partial(split_hands_by_continuity, memory=0.2)
h3 = partial(collapse_trills, max_ioi_s=0.1, min_notes=3)

class Cadena:
    def __init__(self, *fs): self.fs = fs
    def __call__(self, r):
        for f in self.fs:
            r = f(r)
        return r

CONFIGURACIONES = {
    "base, legato 0.5": dict(legato=0.5),
    "legato 0.75": dict(legato=0.75),
    "legato 1.0": dict(legato=1.0),
    "H1": dict(legato=0.75, pre=h1),
    "H2": dict(legato=0.75, hands=h2),
    "H3": dict(legato=0.75, pre=h3),
    "voces MD": dict(legato=0.75, voices=("right",)),
    "H1 + H2": dict(legato=0.75, pre=h1, hands=h2),
    "H1 + H2 + H3": dict(legato=0.75, pre=Cadena(h1, h3), hands=h2),
    "H1 + H2 + voces MD": dict(legato=0.75, pre=h1, hands=h2, voices=("right",)),
    "H1 + H2 + voces ambas manos": dict(legato=0.75, pre=h1, hands=h2, voices=("right", "left")),
}

## 4. Ablación en desarrollo

In [4]:
filas = []
for nombre, kw in CONFIGURACIONES.items():
    for modelo in ("ajustado", "original"):
        b, r, l = bsa_medio(DESARROLLO, modelo, **kw)
        filas.append({"configuracion": nombre, "modelo": modelo, "bsa": b, "bsa_md": r, "bsa_mi": l})
ablacion = pd.DataFrame(filas)
ablacion.to_csv(SALIDA / "oe5_ablacion_desarrollo.csv", index=False)
ablacion.pivot_table(index="configuracion", columns="modelo", values="bsa", sort=False).mul(100).round(2)

modelo,ajustado,original
configuracion,,
"base, legato 0.5",24.57,24.30
legato 0.75,25.22,24.84
legato 1.0,24.93,24.52
H1,25.65,25.56
H2,27.99,27.33
H3,25.53,25.07
voces MD,27.80,27.69
H1 + H2,28.48,28.30
H1 + H2 + H3,27.24,27.96


La mejor combinación en desarrollo es H1 + H2 + voces en la mano derecha con legato 0.75. H3 sube un poco sola pero resta al combinarla, porque también colapsa pasajes rápidos escritos que no son adornos, así que queda fuera. Separar voces en las dos manos baja la izquierda, donde la segunda voz que detecta la heurística casi nunca coincide con la de la edición. Los dos checkpoints quedan prácticamente empatados y se conserva el ajustado, que es el del sistema.

## 5. Evaluación única en prueba

In [5]:
FINAL = CONFIGURACIONES["H1 + H2 + voces MD"]
SIN_VOCES = CONFIGURACIONES["H1 + H2"]
comparadas = {"base, legato 0.5": CONFIGURACIONES["base, legato 0.5"], "H1 + H2": SIN_VOCES, "H1 + H2 + voces MD": FINAL}
filas = []
for nombre, kw in comparadas.items():
    for particion, piezas in (("desarrollo", DESARROLLO), ("prueba", PRUEBA)):
        b, r, l = bsa_medio(piezas, **kw)
        filas.append({"configuracion": nombre, "particion": particion, "bsa": b, "bsa_md": r, "bsa_mi": l})
prueba = pd.DataFrame(filas)
prueba.pivot_table(index="configuracion", columns="particion", values="bsa", sort=False).mul(100).round(2)

particion,desarrollo,prueba
configuracion,,
"base, legato 0.5",24.57,22.56
H1 + H2,28.48,24.47
H1 + H2 + voces MD,29.20,25.47


## 6. Conjunto externo

Tres obras de otros estilos con edición Braille compás sobre compás de la biblioteca BrailleOrch y grabaciones de dominio público o CC0, transcritas en `amt-externo-transcribe`. Für Elise y el vals empiezan con anacrusa, de media negra y de una negra, y se pasa como dato igual que el compás y la armadura.

In [6]:
EXTERNO = {
    "fur_elise": ("Beethoven, Für Elise", "Bor001", 0.5),
    "fille": ("Debussy, La fille aux cheveux de lin", "Bor195", 0.0),
    "vals_chopin": ("Chopin, Vals en la menor B. 150", "Bor343", 1.0),
}
referencias = {}
for clave, (_, codigo, _) in EXTERNO.items():
    z = zipfile.ZipFile(descargar(f"https://www.brailleorch.org/wp-content/uploads/{codigo}.zip", f"{codigo}.zip"))
    brf = next(n for n in z.namelist() if n.lower().endswith(".brf"))
    referencias[clave] = parse_reference(z.read(brf).decode("latin-1"))[0]

filas = []
for nombre, kw in comparadas.items():
    for clave, (obra, _, anacrusa) in EXTERNO.items():
        d = json.loads((NOTAS_EXT / f"ajustado__{clave}.json").read_text())
        ev = evaluate_piece(referencias[clave], d["notas"], d["duracion_s"], pickup_quarters=anacrusa, **kw)
        filas.append({"configuracion": nombre, "particion": obra, "bsa": ev.both.bsa, "bsa_md": ev.right.bsa, "bsa_mi": ev.left.bsa})
externo = pd.DataFrame(filas)
pd.concat([prueba, externo]).to_csv(SALIDA / "oe5_prueba_y_externo.csv", index=False)
externo.pivot_table(index="configuracion", columns="particion", values="bsa", sort=False).mul(100).round(2)

particion,"Beethoven, Für Elise","Debussy, La fille aux cheveux de lin","Chopin, Vals en la menor B. 150"
configuracion,,,
"base, legato 0.5",35.57,14.72,24.08
H1 + H2,38.44,14.36,29.56
H1 + H2 + voces MD,28.55,18.04,24.75


H1 y H2 mejoran en todos los conjuntos. La separación de voces depende del estilo. Ayuda en el contrapunto de Bach y en Debussy, y empeora Für Elise y el vals, que son una melodía con acompañamiento donde la mano derecha casi siempre tiene una sola voz y la heurística inventa una segunda. Por eso el pipeline deja H1, H2 y legato 0.75 activos por omisión y la separación de voces como opción.

## 7. Techo del traductor

La partitura digital de las Open Goldberg Variations, de la misma edición que la referencia Braille, se traduce directamente con el lector de MusicXML del repositorio. No hay audio ni cuantizador, así que lo que falte para el 100 % es diferencia de convención entre el sistema y el transcriptor humano.

In [7]:
mxl = zipfile.ZipFile(descargar("http://open-goldberg.oankali.net/score.mxl", "ogv_score.mxl"))
(DATOS / "ogv_score.xml").write_bytes(mxl.read(next(n for n in mxl.namelist() if not n.startswith("META-INF"))))
partitura = parse_musicxml(str(DATOS / "ogv_score.xml"))

CORTES = [0, 32, 64, 98, 114, 148, 180, 216, 248, 280, 296, 328, 360, 392, 424, 456, 488,
          538, 570, 602, 634, 666, 682, 714, 746, 778, 812, 844, 876, 908, 940, 958, 990]

filas, total = [], None
for i, p in enumerate(goldberg):
    s = Score(beats=p.beats, beat_type=p.beat_type, fifths=p.fifths)
    s.right.measures = partitura.right.measures[CORTES[i]:CORTES[i + 1]]
    s.left.measures = partitura.left.measures[CORTES[i]:CORTES[i + 1]]
    md_, mi_ = translate_score(s, measures_per_line=1)
    ref = reference_streams(p, False)
    r, l = compute_bsa(ref.right, "".join(md_), MAX_CELLS), compute_bsa(ref.left, "".join(mi_), MAX_CELLS)
    ambas = merge(r, l)
    total = ambas if total is None else merge(total, ambas)
    filas.append({"pieza": i, "nombre": nombres[i], "particion": "desarrollo" if i in DESARROLLO else "prueba",
                  "bsa": ambas.bsa, "bsa_md": r.bsa, "bsa_mi": l.bsa})
techo = pd.DataFrame(filas)
techo.to_csv(SALIDA / "oe5_techo_traductor.csv", index=False)
techo.groupby("particion")[["bsa", "bsa_md", "bsa_mi"]].mean().mul(100).round(2)

,bsa,bsa_md,bsa_mi
particion,,,
desarrollo,73.00,70.00,74.58
prueba,67.12,66.19,68.09


In [8]:
pd.DataFrame([{"categoria": c, "ref": s.ref, "gen": s.hyp, "aciertos": s.matches, "exactitud": s.matches / max(s.ref, s.hyp)}
              for c, s in total.per_category.items()]).sort_values("ref", ascending=False).round(4)

,categoria,ref,gen,aciertos,exactitud
3,notas,19469,21385,12602,0.5893
2,octavas,4486,4447,4094,0.9126
0,ligaduras,1694,1648,1548,0.9138
4,in_accords,1414,1682,1274,0.7574
1,alteraciones,1313,1288,1098,0.8363
5,intervalos,270,97,29,0.1074
6,barras,38,0,1,0.0263


In [9]:
techo.sort_values("bsa").head(5).round(4)

,pieza,nombre,particion,bsa,bsa_md,bsa_mi
26,26,Variación 26,prueba,0.2411,0.2388,0.2439
28,28,Variación 28,prueba,0.4843,0.4838,0.4849
23,23,Variación 23,prueba,0.4960,0.4992,0.4925
29,29,Variación 29,prueba,0.4983,0.4447,0.5583
8,8,Variación 8,desarrollo,0.5349,0.5124,0.5586


## 8. Dónde está la brecha

Con la partitura correcta el traductor llega cerca del 70 %. Con el audio, la misma métrica queda alrededor de 25 a 29 %. La diferencia la explica casi toda la etapa acústica y rítmica: notas que el modelo no detecta o agrega, barras que se desplazan porque la intérprete cambia el tempo y el cuantizador usa uno fijo, y duraciones que dependen de la articulación. Las heurísticas recuperan una parte con poco costo, pero cerrar la brecha requiere estimar el pulso compás a compás, lo que queda fuera de este trabajo.

In [10]:
resumen = pd.DataFrame([
    {"condicion": "audio, base", "desarrollo": bsa_medio(DESARROLLO, legato=0.5)[0], "prueba": bsa_medio(PRUEBA, legato=0.5)[0]},
    {"condicion": "audio, configuracion final", "desarrollo": bsa_medio(DESARROLLO, **FINAL)[0], "prueba": bsa_medio(PRUEBA, **FINAL)[0]},
    {"condicion": "partitura de la misma edicion", "desarrollo": techo[techo.particion == "desarrollo"].bsa.mean(), "prueba": techo[techo.particion == "prueba"].bsa.mean()},
])
resumen.set_index("condicion").mul(100).round(2)

,desarrollo,prueba
condicion,,
"audio, base",24.57,22.56
"audio, configuracion final",29.20,25.47
partitura de la misma edicion,73.00,67.12
